# Data: Exploration and Preparation — Quiz

Self-check for Lesson 2. Not graded: answer before expanding, and treat a
question you cannot answer as a pointer back to the handout rather than as a
failure.

Questions marked **(reasoning)** need an argument, not a recalled definition.
Those are the closest thing to the exam you will see before the sample
papers.

## Part 1 — Exploratory analysis and missing values

**1. A systematic first look at a dataset answers four questions. What are they, and why must "where are the gaps" come before you decide how to preprocess anything?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Size and types (<code>.shape</code>, <code>.dtypes</code>); the target's distribution and baseline; where the gaps are; the shape of numeric columns and how they relate to each other and the target.</li>
        <li>It has to come before preprocessing decisions because the missingness <b>mechanism</b> (Part 1, next questions) determines which fix is even valid — deciding to impute before knowing whether a column is missing completely at random (MCAR) or missing at random (MAR) risks silently reweighting the sample.</li>
    </ul>
    </p>
</details>

**2. Looking at the whole dataset — including what will become the test set — during exploratory analysis is safe. Fitting a transform using the whole dataset is not. What is the actual difference between the two?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Looking does not <b>learn</b> a parameter; nothing about a fitted function changes as a result of a human reading a histogram.</li>
        <li>Fitting a transform (a mean, a set of categories, a set of neighbours) produces a number that becomes part of <code>f</code>, and if that number was computed using test rows, <code>f</code> is no longer independent of the test set — handout Section 8.1.</li>
        <li>The boundary is: did this step change what the model does with an input? If yes, and it used test rows to decide that, it is a leak.</li>
    </ul>
    </p>
</details>

**3. Define the three missingness mechanisms — missing completely at random (MCAR), missing at random (MAR) and missing not at random (MNAR) — precisely, in terms of what the probability of a value being missing depends on.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li><b>MCAR</b> — depends on nothing: <code>P(missing) = P(missing | x)</code> for all rows equally.</li>
        <li><b>MAR</b> — depends on an <b>observed</b> variable, not on the missing value itself.</li>
        <li><b>MNAR</b> — depends on the <b>missing value itself</b>, even after conditioning on everything observed.</li>
    </ul>
    </p>
</details>

**4. Give an example of a MAR missingness pattern that is <i>not</i> from the handout, and explain what would have to be true for it to become MNAR instead.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Any answer of the shape: a value is missing more often depending on some OTHER, observed column. Example: exam scores missing more often for students who registered late (an observed registration-date column), not because of the score itself.</li>
        <li>It becomes MNAR the moment the missingness depends on the (unobserved) score itself — e.g. students who failed being disproportionately the ones who never submitted, even after accounting for registration date.</li>
    </ul>
    </p>
</details>

**5. A column is missing completely at random (MCAR) for an 8% fraction <code>p</code> of rows, filled with its own sample mean. Derive why its correlation with another, fully observed column shrinks, and by how much.** <b>(reasoning)</b>

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Var(X') = (1-p)·Var(X): the imputed entries contribute nothing to the sum of squared deviations from the mean.</li>
        <li>Cov(X', Y) = (1-p)·Cov(X, Y): the imputed entries, all equal to a constant, contribute nothing to the covariance beyond what a constant contributes.</li>
        <li>ρ(X', Y) = Cov(X',Y) / (√Var(X')·σ_Y) = (1-p)Cov(X,Y) / (√(1-p)·σ_X·σ_Y) = √(1-p)·ρ(X,Y) — the covariance shrinks linearly in (1-p) but enters the ratio against √Var(X'), leaving a net factor of √(1-p).</li>
        <li>At p=0.08, the attenuation factor is √0.92 ≈ 0.96 — small but real, and larger for columns with more missingness.</li>
    </ul>
    </p>
</details>

## Part 2 — Outliers

**6. Why does adding a missingness indicator column preserve information under MAR that plain mean/median imputation destroys?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Under MAR, the FACT of being missing correlates with an observed variable (e.g. tenure) and is therefore informative.</li>
        <li>Plain imputation replaces the gap with a constant and discards that fact entirely; a binary "was this missing" column keeps it available to the model even after the value itself has been filled in.</li>
    </ul>
    </p>
</details>

**7. The constant 1.5 in Tukey's interquartile range (IQR) fence, <code>[Q1 - 1.5·IQR, Q3 + 1.5·IQR]</code>, is not arbitrary. What is it calibrated against?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Against the standard normal distribution: for X ~ N(μ, σ²), Q1 = μ - 0.6745σ and Q3 = μ + 0.6745σ, so IQR = 1.349σ.</li>
        <li>1.5 was chosen so the resulting fence (roughly μ ± 2.698σ) flags a comparable tail fraction to a 3-sigma z-score rule on a genuinely normal column — the two rules are calibrated to agree with each other under normality.</li>
    </ul>
    </p>
</details>

**8. On real, contaminated data (a column with genuine billing-error outliers), the z-score rule and the IQR rule disagree — the z-score rule typically flags <i>fewer</i> points. Why?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The z-score rule's own ingredients, the mean and standard deviation, are computed from the same contaminated data and are dragged up by the very outliers they are meant to catch — inflating s widens the flagging threshold.</li>
        <li>Q1 and Q3 barely move under the same contamination, since a handful of extreme points cannot shift a quartile unless they exceed 25% of the data — so IQR stays a more stable reference.</li>
    </ul>
    </p>
</details>

**9. A column of "days since signup" contains a few values of -5. Neither the z-score rule nor the IQR rule flags them as outliers. Explain precisely why not, and what kind of check would catch them.** <b>(reasoning)</b>

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Both rules are purely distributional: they flag values that are <b>unusual relative to the rest of the column</b>. If the column's own spread is wide (say 0 to 900), a handful of small negative values are not statistically extreme even though they are impossible.</li>
        <li>Only a <b>domain rule</b> — e.g. "days since signup must be ≥ 0" — checks validity rather than typicality, and catches this.</li>
    </ul>
    </p>
</details>

## Part 3 — Feature scaling

**10. Why does a large ratio between two features' variances slow down (or destabilise) gradient descent when a single global learning rate is used?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>For a quadratic cost, the Hessian of uncorrelated centred features is diag(σ₁², …, σₙ²): the cost surface is stretched along the high-variance axis and squeezed along the low-variance one.</li>
        <li>The safe step size along axis j is roughly 2/σⱼ²; one global rate is bounded by the largest-variance feature, so progress along the smallest-variance direction is painfully slow — or, if the rate is set too large for that axis, the update overshoots and diverges on the steep one.</li>
    </ul>
    </p>
</details>

**11. A colleague standardises the full dataset — computing the mean and standard deviation over train and test together — and only splits afterwards. Explain precisely why this is a leak, even though it "only" touches the scaler, not the model.** <b>(reasoning)</b>

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The fitted scaler's mean and standard deviation are learned parameters, and they were computed using rows that later become the test set.</li>
        <li>Those parameters are part of the function <code>f</code> that eventually makes predictions (handout Section 8.1); since they depended on test rows, <code>f</code> is no longer independent of the test set, and the unbiasedness argument from Lesson 1 no longer holds.</li>
        <li>Nothing about the code looks wrong and no error is raised — which is exactly why this class of leak is dangerous.</li>
    </ul>
    </p>
</details>

**12. What is the condition number κ, and what does standardising uncorrelated features do to it?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>κ = σ²max / σ²min — the ratio of the largest to the smallest feature variance (equivalently, the ratio of the Hessian's largest to smallest eigenvalue).</li>
        <li>Standardising every feature to variance 1 drives κ towards 1, making the cost surface close to a sphere and a single global learning rate near-optimal along every axis at once.</li>
    </ul>
    </p>
</details>

**13. Why is <code>MinMaxScaler</code> generally more sensitive to a single extreme outlier than <code>StandardScaler</code>?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>MinMaxScaler's range is defined directly by x_min and x_max — a single extreme value sets one end of the scale outright, compressing every ordinary value into a sliver of [0, 1].</li>
        <li>StandardScaler is pulled by outliers too, through the mean and standard deviation, but an extreme value's effect on a mean or a standard deviation (averaged over many points) is much more diluted than its effect on a raw min or max.</li>
    </ul>
    </p>
</details>

## Part 4 — Categorical encoding

**14. Prove, in a line or two, why one-hot encoding all k categories of a variable together with a fitted intercept produces a rank-deficient design matrix.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>For every row i, the k dummy columns sum to exactly 1: Σⱼ dummyⱼ⁽ⁱ⁾ = 1 = intercept⁽ⁱ⁾.</li>
        <li>So the intercept column equals the sum of the k dummy columns — a linear combination of them — meaning the k+1 columns together have rank at most k, not k+1: rank-deficient by exactly one.</li>
    </ul>
    </p>
</details>

**15. Why is ordinal encoding inappropriate for an unordered category such as "region" (North, South, East, West, Central)?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Ordinal encoding assigns integers that imply a numeric ordering and equal spacing (0, 1, 2, …).</li>
        <li>For an unordered category this tells any model that, say, North and South are numerically closer than North and West — a meaningless and misleading signal for anything that uses the number arithmetically.</li>
    </ul>
    </p>
</details>

**16. What property of a categorical column makes it a good candidate for target encoding rather than one-hot encoding?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>High cardinality — many distinct levels relative to the number of rows (e.g. <code>zip_code</code> with 493 levels across 2000 rows).</li>
        <li>One-hot encoding such a column would add hundreds of mostly-empty columns (handout Section 6.3's curse-of-dimensionality cost); target encoding compresses it to one numeric column instead.</li>
    </ul>
    </p>
</details>

**17. Derive the exact size of the leak between "leave-in" target encoding (a row's own label included in its category's mean) and the honest "leave-one-out" encoding, and explain why it is worst for small categories.** <b>(reasoning)</b>

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>ȳ_c = [(n_c - 1)·ȳ_c⁽⁻ⁱ⁾ + y_i] / n_c, so ȳ_c - ȳ_c⁽⁻ⁱ⁾ = (y_i - ȳ_c⁽⁻ⁱ⁾) / n_c.</li>
        <li>The gap shrinks as 1/n_c: for large categories one row's own label barely moves the group mean.</li>
        <li>For n_c = 1, the formula degenerates completely — ȳ_c = y_i exactly. The "encoded feature" is not correlated with the label; it IS the label, relabelled as an input.</li>
    </ul>
    </p>
</details>

## Part 5 — Pipelines and leakage in practice

**18. State, in one sentence, the general test for whether a preprocessing step must be fitted inside the training fold rather than once on the whole dataset.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Does fitting this step compute anything from the rows it is given? If yes — a mean, a median, a set of neighbours, a per-category average, a set of bin edges — it must be fitted inside the fold; if it applies a fixed, data-independent rule, it does not.</li>
    </ul>
    </p>
</details>

**19. Why does putting <code>SimpleImputer</code> and <code>StandardScaler</code> inside a single <code>Pipeline</code>, rather than applying them as two separate steps in a notebook, make leakage structurally harder rather than merely "against the house rules"?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>A single <code>Pipeline.fit(X_train, y_train)</code> call is the only place any step's <code>.fit()</code> is ever invoked; no step is called on <code>X_test</code> until <code>.predict()</code>, by which point every learned parameter is already fixed.</li>
        <li>There is no code path left in which a step's fit method could accidentally see test rows — the guarantee comes from the object's structure, not from the author remembering to be careful every time.</li>
    </ul>
    </p>
</details>

**20. In the notebook's <code>KNNImputer</code> demonstration, what specifically made it possible to <i>prove</i> that leakage had occurred, rather than merely suspect it?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>The dataset is synthetic with a known train/test split, so the exact identity of each row's nearest neighbours (found via <code>NearestNeighbors</code> fitted on the full dataset) could be checked against which split each neighbour belonged to.</li>
        <li>98 of 128 training rows with a missing age had at least one test-set row among their five nearest neighbours — traced row by row, not inferred from a score alone.</li>
    </ul>
    </p>
</details>

**21. A model's test AUC — area under the receiver operating characteristic curve — jumps from 0.75 to 0.89 after adding an encoded <code>zip_code</code> feature. Give two hypotheses — one legitimate, one a leak — that could explain this jump, and describe an experiment that would distinguish them.**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Legitimate hypothesis: zip code genuinely correlates with the outcome (e.g. regional economic differences that affect churn).</li>
        <li>Leak hypothesis: the encoding was computed using the whole dataset (including test rows) before splitting, so some test rows' own labels leaked into their own encoded feature value — especially likely if the column has high cardinality (small groups per category).</li>
        <li>Distinguishing experiment: refit the encoding using only the training fold (or a proper cross-fitted <code>TargetEncoder</code> inside a <code>Pipeline</code>) and re-measure test AUC. If the honest version's AUC collapses back towards the no-zip baseline, the original jump was a leak, not a real relationship.</li>
    </ul>
    </p>
</details>

**22. Why does Section 9's leakage discussion say "Lesson 5 returns to leakage a third time, this time for hyperparameter tuning"? What is the connection between tuning and the imputation/encoding leaks in this lesson?**

<details>
<summary>
    <font size='3', color='darkgreen'><b>Answer</b></font>
</summary>
    <p>
    <ul>
        <li>Hyperparameter tuning is itself a step that <b>learns something from data</b> — it selects a setting based on how well candidates score on some data.</li>
        <li>If that data includes the test set (or the same fold used for the final evaluation), the choice of hyperparameter is no longer independent of the evaluation, exactly the same failure mode as an imputer or encoder fitted on the full dataset — the SAME test from Question 18 applies to it too.</li>
    </ul>
    </p>
</details>